In [3]:
import numpy as np
import hashlib
import os
import random

# --- Helper: Circular Left Shift (Bitwise Rotation) ---
def rotate_left_8bit(val, n):
    """Perform a circular left shift on an 8-bit integer."""
    return ((val << n) & 0xFF) | (val >> (8 - n))

# --- Finite Field Math ---
def multiply_ints_as_polynomials(x, y):
    z = 0
    while x != 0:
        if x & 1 == 1:
            z ^= y
        y <<= 1
        x >>= 1
    return z

def number_bits(x):
    nb = 0
    while x != 0:
        nb += 1
        x >>= 1
    return nb

def mod_int_as_polynomial(x, m):
    nbm = number_bits(m)
    while True:
        nbx = number_bits(x)
        if nbx < nbm:
            return x
        mshift = m << (nbx - nbm)
        x ^= mshift

def rijndael_multiplication(x, y):
    z = multiply_ints_as_polynomials(x, y)
    m = int('111000011', 2)
    return mod_int_as_polynomial(z, m)

def rijndael_inverse(x):
    if x == 0:
        return 0
    for y in range(1, 256):
        if rijndael_multiplication(x, y) == 1:
            return y

# --- S-box Construction ---
def compute_values(k1, k2, k3, k4, A, b):
    results = []
    for t in range(256):
        value = ((A**2) * (k1 ^ k2 ^ k3 ^ k4) + t**3 + b) % 257
        results.append(value)
    return results

def dot_product(row, x):
    result = 0
    for i in range(8):
        result ^= ((row >> (7 - i)) & 1) & ((x >> (7 - i)) & 1)
    return result

def affine_transformation(A, x, b):
    y = 0
    for i in reversed(range(8)):
        row = (A >> 8 * i) & 0xff
        bit = dot_product(row, x)
        y ^= (bit << i)
    return y ^ b

def combined_affine_transformation(k1, k2, k3, k4, A, b):
    results = compute_values(k1, k2, k3, k4, A, b)
    unique_values = set(results)
    all_possible_values = set(range(257))
    missing_values = all_possible_values - unique_values

    replacement_results = results[:]
    for i, value in enumerate(replacement_results):
        if value == 256:
            if missing_values:
                replacement_results[i] = missing_values.pop()

    inverse_map = {}
    for value in replacement_results:
        inverse = rijndael_inverse(value)
        inverse_map[value] = inverse

    A_matrix = int('1111100001111100001111100001111110001111110001111110001111110001', 2)
    b_vector = int('01100011', 2)
    transformed_values = []

    for value, inverse in inverse_map.items():
        transformed_value = affine_transformation(A_matrix, inverse, b_vector)
        transformed_values.append(transformed_value)

    return transformed_values

# --- Non-Linearity Logic ---
def compute_walsh_hadamard_transform(bool_func, n):
    size = 2 ** n
    transform = [0] * size
    for x in range(size):
        for y in range(size):
            parity = bin(x & y).count('1') % 2
            transform[x] += (-1) ** (bool_func[y] ^ parity)
    return transform

def compute_non_linearity_of_boolean_function(bool_func, n):
    walsh_transform = compute_walsh_hadamard_transform(bool_func, n)
    max_walsh_value = max(abs(val) for val in walsh_transform)
    return (2 ** (n - 1)) - (max_walsh_value // 2)

def extract_boolean_functions_from_sbox(s_box):
    n = int(np.log2(len(s_box)))
    m = int(np.log2(max(s_box) + 1))
    bool_functions = [[] for _ in range(m)]
    for x in range(len(s_box)):
        output = s_box[x]
        for i in range(m):
            bool_functions[i].append((output >> i) & 1)
    return bool_functions

def compute_non_linearity_for_sbox(s_box):
    n = int(np.log2(len(s_box)))
    bool_functions = extract_boolean_functions_from_sbox(s_box)
    return [compute_non_linearity_of_boolean_function(f, n) for f in bool_functions]

# --- Main Execution ---
if __name__ == "__main__":
    # 1. Generate keys from SHA-256
    dummy_key = os.urandom(32)
    digest = hashlib.sha256(dummy_key).digest()
    indices = random.sample(range(32), 4)
    extracted_bytes = [digest[i] for i in indices]

    # 2. Rotate and assign to k1-k4
    rotated_values = [rotate_left_8bit(val, 3) for val in extracted_bytes]
    k1, k2, k3, k4 = rotated_values

    # 3. Automatically determine b
    b_auto = max(k1, k2, k3, k4)

    print(f"Generated Keys (hex): k1={k1:02X}, k2={k2:02X}, k3={k3:02X}, k4={k4:02X}")
    print(f"Automatically selected b: {b_auto:02X}")

    try:
        A_input = int(input("Enter the value of A (hexadecimal, e.g., 0x5E): "), 16)

        # 4. Generate S-box
        s_box = combined_affine_transformation(k1, k2, k3, k4, A_input, b_auto)

        # Display S-Box in 16x16
        print("\nS-box (16x16 Table):")
        for row in range(16):
            for col in range(16):
                print(f"{s_box[row * 16 + col]:02X}", end=' ')
            print()

        # 5. Compute and print non-linearities
        non_linearities = compute_non_linearity_for_sbox(s_box)
        print("\nNon-linearities of all Boolean functions:", non_linearities)

        average_all_boolean_functions = sum(non_linearities) / len(non_linearities)
        print(f"Average non-linearity of all Boolean functions: {average_all_boolean_functions}")

    except ValueError:
        print("Please enter a valid hexadecimal value for A!")

Generated Keys (hex): k1=55, k2=07, k3=F5, k4=04
Automatically selected b: F5
Enter the value of A (hexadecimal, e.g., 0x5E): 4F

S-box (16x16 Table):
A9 70 41 86 3E 42 1A 69 3C 6E C3 FC EE E7 E6 53 
CF 21 22 CD 0E 33 BE 17 24 74 8C 18 94 0F 1E 14 
C1 FB E4 F4 0B 64 D5 2E 98 DC AF A2 DF E9 F2 30 
D4 E8 8D 4F EC 85 38 D9 68 2C 08 BA 89 E5 C0 58 
7B 5C 4C 6F 5A EF A6 F5 CC 59 63 9C A7 95 25 DB 
48 B3 0C 61 84 D2 DE 73 20 F8 2B 65 26 91 DA D8 
6B 07 2A B6 55 A5 CA E1 99 82 23 C7 87 32 56 77 
4D AE 02 10 54 62 A0 03 AB 72 75 29 F0 B2 71 36 
50 45 83 16 5F F9 93 BC 81 2F 1C FE 8A 80 4E F7 
37 78 A4 AD 46 3F 76 9F 5D B1 C5 12 52 3B 40 F3 
7A 31 A8 47 F1 88 60 96 0A D1 B8 BD 01 44 6A 7D 
0D 13 05 B9 43 B0 AA 3D 49 2D A3 DD D7 7E 3A 97 
8B 90 C9 8F C2 B4 A1 9D 9B 66 34 CB 15 E2 D3 5B 
9A B7 6D EA 06 1B ED EB E0 39 BB C8 CE C4 FD 8E 
E3 F6 FA BF 6C 5E 4A AC 57 FF 79 D0 19 27 1F 9E 
09 D6 7F 4B 92 04 28 67 1D 35 11 00 51 7C C6 B5 

Non-linearities of all Boolean functions: [104, 104, 104, 104, 1